In [1]:
# baseline libraries
import numpy as np
import pandas as pd

# preprocessing libraries
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer

# pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ML models
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB


In [2]:
#fix random_state for reproducibility
RANDOM_STATE = 42

# import development and evaluation set
development_set = pd.read_csv("development.csv")
evaluation_set = pd.read_csv("evaluation.csv")

In [3]:
def global_preprocessing(df):
    """
    Function that performs the global preprocessing,
    i.e. all those steps common to the whole dataset, and therefore not involving splits.
    ---
    INPUT:
    - raw df
    OUTPUT:
    - features preprocessed
    - labels
    - ids
    """
    #copy 
    df = df.copy()
    ids = df["Id"]
    
    if "Id" in df.columns:
        df.set_index("Id", inplace=True)

    # Drop timestamp
    if "timestamp" in df.columns:
        df.drop(columns=["timestamp"], inplace=True)


    #fillna within documents
    df[["title", "article"]] = df[["title", "article"]].fillna("")
    # Feature engineering
    df["title_length"] = df["title"].apply(lambda x: len(str(x).split())) 
    df["article_length"] = df["article"].apply(lambda x: len(str(x).split())) #compute article length

    #columns to keep
    cols = ["source", "title", "article", "title_length", "article_length", "page_rank"]

    if "label" in df.columns:
        X = df[cols]
        y = df["label"]
        return X, y, ids

    else:
        X = df[cols]
        return X, None, ids

# SVM - BEST MODEL

In [4]:
# ============================================================
# 1 - Global preprocessing
# ============================================================

# return the FULL development set, with column preprocessed
X_dev_preproc, y_dev, _ = global_preprocessing(development_set)

In [5]:
# ============================================================
# 2 - Declare Best Hyperparameters
# ============================================================

# Customized stop word list
s_words = list(text.ENGLISH_STOP_WORDS) # full default list of 150 words, i.e. the list called when we use stop_words=["english"] within TfidfVec
s_words_firstcap = [w.capitalize() for w in s_words] #like s_words, but with Capital starting letter
# complete list of words to delete
stop_words_full = s_words + s_words_firstcap

# SVM - best hyperparameters
svm_params = {
    "C": 0.1, # regularization parameter
    "class_weight": "balanced", # adjust weights inversely proportional to class frequencies in the input data
    "loss": "squared_hinge", # loss function
    "max_iter": 10000, 
    "random_state" : RANDOM_STATE
}

# TITLE VECTORIZER - best hyperparameters
svm_title_vect_params = {
    "strip_accents" : "ascii", # remove accents and perform other character normalization
    "lowercase" : False, #keep all characters as they are before tokenizing
    "ngram_range" : (1,3), # the lower and upper boundary of the range of n-values for different n-grams to be extracted
    "max_df" : 0.5, # ignore terms that have a document frequency strictly higher than the given threshold
    "min_df": 2, #ignore terms that have a document frequency strictly lower than the given threshold
    "stop_words" : stop_words_full, # remove stop words, customized list
    "sublinear_tf" : False # True for wide lenght range documents
}

# ARTICLE VECTORIZER - best hyperparameters
svm_article_vect_params = {
    "strip_accents": 'ascii',
    "lowercase": False,
    "ngram_range": (1, 2),
    "max_df": 0.6, 
    "min_df": 15, #ignore terms that have a document frequency strictly lower than the given threshold
    "sublinear_tf": True, #apply sublinear tf scaling, i.e. replace tf with 1 + log(tf). True for wide lenght range documents
    "stop_words" : stop_words_full 
}

In [6]:
# ============================================================
# 3 - Pipeline
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("title", TfidfVectorizer(**svm_title_vect_params), "title"),
        ("article", TfidfVectorizer(**svm_article_vect_params), "article"), 
        ("source", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist"), ["source"]),
        ("ordinal_features", MinMaxScaler(), ["page_rank"]), #scaling required for SVM. Min Max to preserve the skewed distribution
        ("continuous_features", MinMaxScaler(), ["title_length", "article_length"]) #scaling required since SVM relies on distances. Min Max to preserve the skewed distribution
    ]
)

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(**svm_params))
])

In [8]:
# ============================================================
# 4 - Prediction
# ============================================================

#fit on the whole development set
svm_pipeline.fit(X_dev_preproc, y_dev)

#global preprocessing on evaluation set
X_eval_preproc, _, ids_eval = global_preprocessing(evaluation_set) 

#predict 
y_pred = svm_pipeline.predict(X_eval_preproc)

#create csv
submission = pd.DataFrame({
    "Id" : ids_eval,
    "Predicted": y_pred
})
submission.to_csv("357948_submission_svm.csv", index=False)

# CNB - BEST MODEL

In [9]:
# ============================================================
# 1 - Global preprocessing
# ============================================================

# return the FULL development set, with column preprocessed
X_dev_preproc, y_dev, _ = global_preprocessing(development_set)

In [10]:
# ============================================================
# 2 - Declare Best Hyperparameters
# ============================================================

# Customized stop word list
s_words = list(text.ENGLISH_STOP_WORDS) # full default list of 150 words, i.e. the list called when we use stop_words=["english"] within TfidfVec
s_words_firstcap = [w.capitalize() for w in s_words] #like s_words, but with Capital starting letter
# complete list of words to delete
stop_words_full = s_words + s_words_firstcap

# NB - best hyperparameters
nb_params = {
    "alpha": 0.15, #additive smoothing parameter
    "norm" : True # whether or not a second normalization of the weights is performed
}

# TITLE VECTORIZER - best hyperparameters
nb_title_vect_params = {
    "lowercase" : False, #keep all characters as they are before tokenizing
    "strip_accents" : "ascii",
    "ngram_range" : (1,3), # the lower and upper boundary of the range of n-values for different n-grams to be extracted
    "min_df" : 1, #ignore terms that have a document frequency strictly lower than the given threshold
    "use_idf" : True, # enable inverse-document-frequency reweighting. If False, idf(t) = 1
    "stop_words" : stop_words_full, # remove stop words, customized list
    "sublinear_tf" : False,
    "norm" : "l2"
}

# ARTICLE VECTORIZER - best hyperparameters
nb_article_vect_params = {
    "lowercase" : False, #keep all characters as they are before tokenizing
    "strip_accents" : "ascii",
    "ngram_range" : (1,3), # the lower and upper boundary of the range of n-values for different n-grams to be extracted
    "min_df" : 3, #ignore terms that have a document frequency strictly lower than the given threshold
    "use_idf" : False, # enable inverse-document-frequency reweighting. If False, idf(t) = 1
    "stop_words" : stop_words_full, # remove stop words, customized list
    "sublinear_tf" : True,
    "norm" : "l2"
}

In [11]:
# ============================================================
# 3 - Pipeline
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("title", TfidfVectorizer(**nb_title_vect_params), "title"),
        ("article", TfidfVectorizer(**nb_article_vect_params), "article"), 
        ("source", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist"), ["source"])
    ]
)

nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", ComplementNB(**nb_params))
])

In [12]:
# ============================================================
# 4 - Prediction
# ============================================================

#fit on the whole development set
nb_pipeline.fit(X_dev_preproc, y_dev)

#global preprocessing on evaluation set
X_eval_preproc, _, ids_eval = global_preprocessing(evaluation_set) 

#predict 
y_pred = nb_pipeline.predict(X_eval_preproc)

#create csv
submission = pd.DataFrame({
    "Id" : ids_eval,
    "Predicted": y_pred
})
submission.to_csv("357948_submission_cnb.csv", index=False)